# 19. HS 해설서 + 과거 판정 사례 통합 검색 모델

18번 최종 모델의 평가 대상, 전처리, 4-fold StratifiedGroupKFold 분할을 그대로 재현하고 HS 지식베이스 검색을 최종 HS 코드 후보 선정에 연결합니다.

비교 모델:
- TF-IDF + LinearSVC
- SBERT product_only
- SBERT product_plus_ruling
- SBERT product_plus_ruling_stopword
- SBERT KB_only
- SBERT product_only + KB
- SBERT product_plus_ruling_stopword + KB

해설서는 고정된 외부 분류 지식베이스로 검색합니다. 과거 판정 사례 검색은 Fold의 학습 데이터만 사용합니다. 두 검색 결과는 서로 다른 코퍼스의 유사도 점수를 직접 더하지 않고 Reciprocal Rank Fusion(RRF)으로 결합합니다.

해설서의 HS 코드가 호(4단위) 등 상위 단위로 기록된 경우 평가 데이터의 세부 코드와 접두어 기준으로 연결합니다. 해설서 검색 결과만으로 법적 분류를 확정하지 않으며, 분류 근거를 검토하기 위한 후보 검색으로 해석합니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy

In [ ]:
import re,hashlib,json
import numpy as np
import pandas as pd
from urllib.parse import quote
from pathlib import Path
from kiwipiepy import Kiwi
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,f1_score,classification_report
from scipy.stats import binomtest

BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
DATA_FILE=quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
df=pd.read_csv(BASE+DATA_FILE,encoding='utf-8-sig').fillna('')
kb=pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
INPUT_COLUMNS=['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
TARGET='HS Code'
SUMMARY='Preprocessed Ruling Summary'
STOPWORDS={'것','경우','관세율표','규정','기재','등','따라서','물품','및','본건','본품','분류','설명','소호','수','이상','제품','함','해당','호','표','관세'}
for col in INPUT_COLUMNS+[TARGET,SUMMARY]: df[col]=df[col].astype(str).str.strip()
df=df[df[TARGET].ne('')].reset_index(drop=True)
df['query_text']=df[INPUT_COLUMNS].agg(' '.join,axis=1).str.strip()
df['model_text']='[PRODUCT] '+df['Product Name']+' [INGREDIENTS_KR] '+df['Main Ingredients (KR)']+' [INGREDIENTS_EN] '+df['Main Ingredients (EN)']+' [FORM] '+df['Product Form and Packaging']+' [USE] '+df['Intended Use']
df['duplicate_group']=pd.util.hash_pandas_object(df[INPUT_COLUMNS],index=False).astype(str)
class_counts=df[TARGET].value_counts()
eligible_labels=class_counts[class_counts>=4].index
eval_df=df[df[TARGET].isin(eligible_labels)].copy()
group_counts=eval_df.groupby(TARGET)['duplicate_group'].nunique()
eligible_labels=group_counts[group_counts>=4].index
eval_df=eval_df[eval_df[TARGET].isin(eligible_labels)].reset_index(drop=True)
X=eval_df['model_text'].reset_index(drop=True)
y=eval_df[TARGET].astype(str).reset_index(drop=True)
groups=eval_df['duplicate_group'].reset_index(drop=True)
labels=np.sort(y.unique())
cv=StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=42)
splits=list(cv.split(X,y,groups))
print('평가 사례:',len(eval_df),'| HS 코드:',len(labels),'| KB 문서:',len(kb))
print('Fold 크기:',[len(te) for _,te in splits])

## 1. 해설서 지식베이스 전처리 및 임베딩

기존 16번 실험과 동일하게 `document_text`, `hs_code`, `rule_type`, `source_sheet`를 사용합니다. 제품 정보는 검색 질의로 사용하고, 해설서 문서에 정답 판정요지를 추가하지 않습니다.

In [ ]:
kiwi=Kiwi()
def remove_stopword_morphs(text):
    chars=list(str(text))
    for token in kiwi.tokenize(str(text)):
        if token.form in STOPWORDS:
            for i in range(token.start,token.start+token.len): chars[i]=''
    return ''.join(chars)

for col in ['document_text','hs_code','rule_type','source_sheet','knowledge_id']:
    if col not in kb.columns: kb[col]=''
    kb[col]=kb[col].fillna('').astype(str).str.strip()
kb['document_text_sw']=kb['document_text'].map(remove_stopword_morphs)
kb['search_text_base']=(kb['document_text']+' '+kb['rule_type']+' '+kb['source_sheet']).str.strip()
kb['search_text_sw']=(kb['document_text_sw']+' '+kb['rule_type']+' '+kb['source_sheet']).str.strip()
kb['hs_digits']=kb['hs_code'].map(lambda x:re.sub(r'\D','',str(x)))
kb_code=kb[kb['hs_digits'].ne('')].reset_index(drop=True)
print('전체 KB 문서:',len(kb),'| HS 코드가 명시된 문서:',len(kb_code))
display(kb.groupby(['source_sheet','rule_type']).size().rename('문서 수').reset_index().head(20))
display(kb_code[['knowledge_id','hs_code','rule_type','source_sheet','document_text']].head(8))

## 2. E5 임베딩 및 후보 코드 매핑

해설서 코드와 평가 데이터의 HS 코드가 완전히 같지 않을 수 있으므로, 코드 숫자열의 접두어 관계를 이용해 상위 호·소호 근거를 세부 후보 코드에 연결합니다. 코드가 비어 있는 총설·공통 규정 문서는 검색 근거로는 확인할 수 있지만 특정 HS 코드 후보 점수에는 직접 반영하지 않습니다.

In [ ]:
MODEL_NAME='intfloat/multilingual-e5-base'
sbert=SentenceTransformer(MODEL_NAME)
def encode_docs(texts):
    return sbert.encode(['passage: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def encode_queries(texts):
    return sbert.encode(['query: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def code_digits(value): return re.sub(r'\D','',str(value))
def code_compatible(a,b):
    a,b=code_digits(a),code_digits(b)
    return bool(a and b and (a.startswith(b) or b.startswith(a)))
def make_vectorizer():
    return FeatureUnion([
        ('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),
        ('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))
    ])
kb_emb_base=encode_docs(kb_code['search_text_base'].tolist())
kb_emb_sw=encode_docs(kb_code['search_text_sw'].tolist())
query_embs=encode_queries(eval_df['query_text'].tolist())
print('임베딩 모델:',MODEL_NAME,'| 장치:',sbert.device)

## 3. 검색 순위 결합 함수

과거 사례 코퍼스와 해설서 코퍼스의 cosine similarity 분포가 다를 수 있어 원점수의 단순 가중합은 사용하지 않습니다. 각 코퍼스에서 HS 코드별 최고 유사도 순위를 구한 뒤 RRF 점수를 계산합니다. 기본 RRF 상수는 60이며 두 검색 경로에 동일한 가중치를 부여합니다.

In [ ]:
RRF_K=60
def rank_case_codes(scores,train):
    tmp=pd.DataFrame({'hs_code':train[TARGET].astype(str).to_numpy(),'score':scores})
    return tmp.groupby('hs_code')['score'].max().sort_values(ascending=False)
def rank_kb_codes(scores):
    code_scores={}
    for j,doc_code in enumerate(kb_code['hs_digits']):
        if not doc_code: continue
        for code in labels:
            if code_compatible(doc_code,code):
                code_scores[code]=max(code_scores.get(code,-np.inf),float(scores[j]))
    return pd.Series(code_scores,dtype=float).sort_values(ascending=False)
def rrf_rank(case_rank,kb_rank):
    case_pos={str(code):i+1 for i,code in enumerate(case_rank.index)}
    kb_pos={str(code):i+1 for i,code in enumerate(kb_rank.index)}
    vals={str(code):(1/(RRF_K+case_pos.get(str(code),100000))+1/(RRF_K+kb_pos.get(str(code),100000))) for code in labels}
    return pd.Series(vals).sort_values(ascending=False)
def top_codes(rank,n=3):
    return [str(x) for x in rank.index[:n]]
def make_row(row_index,fold,true_hs,model_name,rank,extra=None):
    rank=rank[rank.index.astype(str).isin(labels)]
    result={'row_index':int(row_index),'fold':int(fold),'true_hs':str(true_hs),'model':model_name,'predicted_hs':str(rank.index[0]) if len(rank) else '', 'top3_candidates':top_codes(rank)}
    if extra: result.update(extra)
    return result

## 4. 동일 Fold에서 기존 모델과 통합 모델 평가

18번과 동일한 4-fold 분할을 사용합니다. 과거 사례 검색 문서는 각 Fold의 학습 사례만 포함합니다. 해설서 지식베이스는 고정 문서이며, 평가 사례의 정답 판정요지를 검색 질의에 넣지 않습니다.

In [ ]:
prediction_rows=[]
fold_rows=[]
kb_evidence_rows=[]
for fold,(tr,te) in enumerate(splits,1):
    train,test=eval_df.iloc[tr].copy(),eval_df.iloc[te].copy()
    fold_rows.append({'fold':fold,'train_n':len(tr),'test_n':len(te),'group_overlap':len(set(groups.iloc[tr])&set(groups.iloc[te]))})
    vectorizer=make_vectorizer()
    x_train=vectorizer.fit_transform(X.iloc[tr])
    x_test=vectorizer.transform(X.iloc[te])
    svm=LinearSVC(C=1.0,class_weight='balanced').fit(x_train,y.iloc[tr])
    svm_scores=svm.decision_function(x_test)
    for local_i,global_i in enumerate(te):
        order=np.argsort(svm_scores[local_i])[::-1]
        svm_rank=pd.Series(svm_scores[local_i],index=svm.classes_).sort_values(ascending=False)
        prediction_rows.append(make_row(global_i,fold,y.iloc[global_i],'TFIDF_LinearSVC',svm_rank))
    doc_texts={
        'SBERT_product_only':train['query_text'].tolist(),
        'SBERT_product_plus_ruling':(train['query_text']+' '+train[SUMMARY]).str.strip().tolist(),
        'SBERT_product_plus_ruling_stopword':(train['query_text']+' '+train[SUMMARY].map(remove_stopword_morphs)).str.strip().tolist()
    }
    doc_embeddings={name:encode_docs(texts) for name,texts in doc_texts.items()}
    for global_i in te:
        row=eval_df.iloc[global_i]
        q=query_embs[global_i]
        case_ranks={}
        for name,emb in doc_embeddings.items():
            scores=emb@q
            rank=rank_case_codes(scores,train)
            case_ranks[name]=rank
            prediction_rows.append(make_row(global_i,fold,row[TARGET],name,rank))
        kb_scores_base=kb_emb_base@q
        kb_scores_sw=kb_emb_sw@q
        kb_rank_base=rank_kb_codes(kb_scores_base)
        kb_rank_sw=rank_kb_codes(kb_scores_sw)
        fallback=case_ranks['SBERT_product_only']
        for variant,kb_rank,kb_scores in [('base',kb_rank_base,kb_scores_base),('stopword',kb_rank_sw,kb_scores_sw)]:
            if global_i==te[0]:
                pass
            for j in np.argsort(kb_scores)[::-1][:5]:
                kb_evidence_rows.append({'row_index':int(global_i),'fold':fold,'knowledge_id':kb_code.iloc[j]['knowledge_id'],'hs_code':kb_code.iloc[j]['hs_code'],'rule_type':kb_code.iloc[j]['rule_type'],'source_sheet':kb_code.iloc[j]['source_sheet'],'similarity':float(kb_scores[j]),'variant':variant,'document_text':kb_code.iloc[j]['document_text']})
        kb_only_rank=kb_rank_base if len(kb_rank_base) else fallback
        prediction_rows.append(make_row(global_i,fold,row[TARGET],'SBERT_KB_only',kb_only_rank,{'kb_fallback':not bool(len(kb_rank_base))}))
        prediction_rows.append(make_row(global_i,fold,row[TARGET],'SBERT_product_only_plus_KB',rrf_rank(fallback,kb_rank_base)))
        prediction_rows.append(make_row(global_i,fold,row[TARGET],'SBERT_product_plus_ruling_stopword_plus_KB',rrf_rank(case_ranks['SBERT_product_plus_ruling_stopword'],kb_rank_base)))
        prediction_rows.append(make_row(global_i,fold,row[TARGET],'SBERT_product_only_plus_KB_stopword',rrf_rank(fallback,kb_rank_sw)))
        prediction_rows.append(make_row(global_i,fold,row[TARGET],'SBERT_product_plus_ruling_stopword_plus_KB_stopword',rrf_rank(case_ranks['SBERT_product_plus_ruling_stopword'],kb_rank_sw)))
predictions=pd.DataFrame(prediction_rows)
fold_audit=pd.DataFrame(fold_rows)
kb_evidence=pd.DataFrame(kb_evidence_rows)
display(fold_audit)
if fold_audit['group_overlap'].sum()!=0: raise ValueError('Train/test 그룹 중복 발생')
print('OOF 예측 수:',len(predictions),'| 모델별 예측 수:',predictions.groupby('model').size().to_dict())
print('해설서 근거 저장 수:',len(kb_evidence))

## 5. OOF 성능 비교

Top-3는 검색 모델의 상위 서로 다른 HS 코드 후보를 기준으로 계산합니다. 해설서 검색에서 코드가 확인되지 않는 경우 KB-only 모델은 product_only 검색으로 대체하며, 해당 건수는 별도로 집계합니다.

In [ ]:
def metric_summary(part):
    yt=part['true_hs'].astype(str)
    yp=part['predicted_hs'].astype(str)
    top3=np.mean([str(t) in [str(x) for x in c] for t,c in zip(yt,part['top3_candidates'])])
    return {'n':len(part),'accuracy':accuracy_score(yt,yp),'macro_f1':f1_score(yt,yp,labels=labels,average='macro',zero_division=0),'weighted_f1':f1_score(yt,yp,average='weighted',zero_division=0),'top3_accuracy':top3}
summary=pd.DataFrame([{'model':name,**metric_summary(part)} for name,part in predictions.groupby('model')]).sort_values('macro_f1',ascending=False)
display(summary.round(4))
fold_metrics=pd.DataFrame([{'model':name,'fold':fold,**metric_summary(part)} for (name,fold),part in predictions.groupby(['model','fold'])])
display(fold_metrics.pivot(index='fold',columns='model',values='accuracy').round(4))
display(fold_metrics.pivot(index='fold',columns='model',values='macro_f1').round(4))
display(predictions[predictions['model'].eq('SBERT_KB_only')]['kb_fallback'].value_counts(dropna=False))

## 6. 모델 간 paired 비교 및 클래스별 성능

동일한 평가 사례의 정오답 변화와 HS 코드별 F1을 확인합니다. 여러 모델 쌍의 검정 결과는 탐색적 참고값으로 해석합니다.

In [ ]:
wide=predictions.pivot(index='row_index',columns='model',values='predicted_hs')
truth=predictions.drop_duplicates('row_index').set_index('row_index')['true_hs'].astype(str)
comparison_rows=[]
from itertools import combinations
for a,b in combinations(wide.columns,2):
    pa,pb=wide[a].astype(str),wide[b].astype(str)
    ac,bc=pa.eq(truth),pb.eq(truth)
    a_only=int((ac&~bc).sum())
    b_only=int((~ac&bc).sum())
    p=binomtest(min(a_only,b_only),a_only+b_only,.5).pvalue if a_only+b_only else 1.0
    comparison_rows.append({'model_a':a,'model_b':b,'both_correct':int((ac&bc).sum()),'a_only_correct':a_only,'b_only_correct':b_only,'both_wrong':int((~ac&~bc).sum()),'mcnemar_exact_p':p})
paired_tests=pd.DataFrame(comparison_rows)
display(paired_tests.sort_values('mcnemar_exact_p').head(20).round(4))
class_rows=[]
for model_name,part in predictions.groupby('model'):
    report=classification_report(part['true_hs'].astype(str),part['predicted_hs'].astype(str),labels=labels,output_dict=True,zero_division=0)
    for code in labels:
        class_rows.append({'model':model_name,'hs_code':code,'precision':report[str(code)]['precision'],'recall':report[str(code)]['recall'],'f1':report[str(code)]['f1-score'],'support':report[str(code)]['support']})
class_metrics=pd.DataFrame(class_rows)
display(class_metrics.pivot(index='hs_code',columns='model',values='f1').round(3))

## 7. 해설서 근거 사례 점검

최종 후보 코드별 근거를 확인할 수 있도록 평가 사례마다 상위 5개 해설서 문서를 저장합니다. 문서의 HS 코드가 공란인 총설·공통 규정은 별도 검색 근거로 추가 검토할 수 있지만, 이번 코드 후보 점수에는 직접 포함하지 않습니다.

In [ ]:
display(kb_evidence.sort_values(['row_index','similarity'],ascending=[True,False]).groupby('row_index').head(5).head(30))
display(kb_code.groupby(['hs_code','rule_type']).size().rename('문서 수').reset_index().sort_values('문서 수',ascending=False).head(30))

## 8. 결과 저장 및 해석 주의사항

- 18번과 동일한 평가 데이터 필터 및 Fold를 사용합니다.
- 해설서 검색 문서는 평가 Fold와 독립된 고정 지식베이스입니다.
- 과거 판정 사례는 각 Fold의 학습 데이터만 검색합니다.
- RRF는 두 검색 경로의 순위를 결합하며, 원점수의 직접 합산을 피합니다.
- 해설서 상위 코드와 세부 HS 코드의 접두어 매핑은 후보 생성용 근사 규칙입니다. 법적 분류 판단에는 해당 호·소호의 문언, 주, 제외 규정 및 통칙을 별도로 검토해야 합니다.
- 최종 성능은 노트북을 실행한 뒤 산출됩니다. 현재 저장소에 결과 수치를 미리 기재하지 않습니다.

In [ ]:
OUT=Path('/content/hs_explanatory_note_integration')
OUT.mkdir(parents=True,exist_ok=True)
predictions.to_csv(OUT/'19_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'19_model_comparison.csv',index=False,encoding='utf-8-sig')
fold_metrics.to_csv(OUT/'19_fold_metrics.csv',index=False,encoding='utf-8-sig')
paired_tests.to_csv(OUT/'19_paired_model_tests.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(OUT/'19_class_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv(OUT/'19_fold_audit.csv',index=False,encoding='utf-8-sig')
kb_evidence.to_csv(OUT/'19_kb_evidence.csv',index=False,encoding='utf-8-sig')
print('19번 결과 저장 완료:',OUT)
import shutil
shutil.make_archive('/content/19_explanatory_note_integration','zip',OUT)